# Build Benchmark Dataset

Reads every unique article from `triage_results.db`, annotates each with a
`should_keep` ground-truth label using GPT-4o, and writes `benchmark.csv`.

**Design choices:**
- `benchmark.csv` contains only ground truth — no pipeline scores.
- Annotation is content-only: publication date is ignored so freshness
  thresholds can be tested independently in `evaluate.ipynb`.
- `confidence` field flags borderline cases for manual review.
- Saves after every article so it is safe to interrupt and resume.

In [1]:
import json
import sqlite3
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

BENCHMARK_PATH = Path('benchmark.csv')
MODEL = 'gpt-5.4'
SLEEP_BETWEEN_CALLS = 0.3

In [2]:
# Load all unique articles from DB — content fields only, no pipeline scores
conn = sqlite3.connect('triage_results.db')
df_raw = pd.read_sql("""
    SELECT t.article_id, t.source, t.title, t.body, t.published_at
    FROM triage_results t
    INNER JOIN (
        SELECT article_id, MAX(id) AS max_id
        FROM triage_results
        GROUP BY article_id
    ) latest ON t.id = latest.max_id
    ORDER BY t.article_id
""", conn)
conn.close()

print(f'Unique articles: {len(df_raw)}')
print(df_raw['source'].value_counts())

Unique articles: 113
source
reddit          68
ars-technica    45
Name: count, dtype: int64


In [4]:
ANNOTATION_PROMPT = """\
You are building a ground-truth benchmark for an enterprise IT newsfeed.
Decide whether this article SHOULD appear in a newsfeed for enterprise IT
managers and security teams.

KEEP if the article reports on (regardless of publication date):
- A confirmed security incident, breach, or active exploitation
  (CVE, ransomware, supply-chain attack, malware campaign, nation-state activity)
- A widespread service outage or infrastructure failure affecting multiple organisations
- A severe bug, broken update, or broken patch with broad enterprise impact
- An official vendor advisory, deprecation, or forced migration with operational impact
- Actionable threat intelligence — even if informally written, as long as it describes
  a confirmed, named, widespread incident or attack pattern

DISCARD if the article is mainly:
- An individual troubleshooting question (single user or single organisation)
- A personal rant, opinion, career advice, or meta-community post
- A shopping guide, product comparison, or generic how-to with no incident
- Geopolitical news without direct enterprise IT operational impact
- Consumer-device news (phones, home routers) with no clear enterprise exposure
- Cryptocurrency-specific incidents with no enterprise software or infrastructure angle
- An anecdote that does not confirm a broader named pattern

IMPORTANT: Ignore publication date entirely. Judge content quality only.

Return JSON only:
{{"should_keep": true, "confidence": "90", "reason": "one sentence"}}
confidence: 0 - 100,
Source: {source}
Title: {title}
Body: {body}
"""


def annotate(source: str, title: str, body: str) -> dict:
    prompt = ANNOTATION_PROMPT.format(
        source=source,
        title=title,
        body=(body or '')[:1000],
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Return only valid JSON. No markdown, no comments.'},
            {'role': 'user',   'content': prompt},
        ],
        response_format={
            'type': 'json_schema',
            'json_schema': {
                'name': 'annotation',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'should_keep': {'type': 'boolean'},
                        'confidence':  {'type': 'integer', 'minimum': 0, 'maximum': 100},
                        'reason':      {'type': 'string'},
                    },
                    'required': ['should_keep', 'confidence', 'reason'],
                    'additionalProperties': False,
                },
            },
        },
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


print('Annotator ready.')

Annotator ready.


In [5]:
# Load existing benchmark if present (incremental — skips already-annotated rows)
if BENCHMARK_PATH.exists():
    existing = pd.read_csv(BENCHMARK_PATH)
    done_ids = set(existing['article_id'])
    print(f'Resuming — {len(done_ids)} already annotated.')
else:
    existing = pd.DataFrame()
    done_ids = set()

to_annotate = df_raw[~df_raw['article_id'].isin(done_ids)].reset_index(drop=True)
print(f'Articles to annotate: {len(to_annotate)}')

Articles to annotate: 113


In [6]:
# Annotate and save — benchmark.csv contains ground truth only
new_rows = []
errors   = []

for i, row in to_annotate.iterrows():
    try:
        result = annotate(
            source=row['source'],
            title=row['title'],
            body=row['body'] or '',
        )
        record = {
            'article_id':        row['article_id'],
            'source':            row['source'],
            'title':             row['title'],
            'body':              (row['body'] or ''),
            'published_at':      row['published_at'],
            'should_keep':       result['should_keep'],
            'confidence':        result['confidence'],
            'annotation_reason': result['reason'],
        }
        new_rows.append(record)

        combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
        combined.to_csv(BENCHMARK_PATH, index=False)

        mark = '\u2713' if result['should_keep'] else '\u2717'
        print(f"[{i+1}/{len(to_annotate)}] {mark} [{result['confidence']:6}] {row['title'][:70]}")

    except Exception as e:
        errors.append({'article_id': row['article_id'], 'error': str(e)})
        print(f"[{i+1}/{len(to_annotate)}] ERROR: {row['title'][:60]} — {e}")

    time.sleep(SLEEP_BETWEEN_CALLS)

print(f'\nDone. Annotated {len(new_rows)} articles. Errors: {len(errors)}')
if errors:
    print('Failed:', errors)

[1/113] ✓ [    92] 5 plead guilty to laptop farm and ID theft scheme to land North Korean
[2/113] ✗ [    78] Critics scoff after Microsoft warns AI feature can infect machines and
[3/113] ✗ [    95] Oops. Cryptographers cancel election results after losing decryption k
[4/113] ✗ [    95] Researchers question Anthropic claim that AI-assisted attack was 90% a
[5/113] ✗ [    96] This hacker conference installed a literal antivirus monitoring system
[6/113] ✓ [    95] How to know if your Asus router is one of thousands hacked by China-st
[7/113] ✓ [    95] Admins and defenders gird themselves against maximum-severity server v
[8/113] ✓ [    92] Browser extensions with 8 million users collect extended AI conversati
[9/113] ✗ [    34] Fraudulent gambling network may actually be something more nefarious
[10/113] ✓ [    92] Microsoft will finally kill obsolete cipher that has wreaked decades o
[11/113] ✓ [    84] Supply chains, AI, and the cloud: The biggest failures (and one succes
[12/113] ✓

In [7]:
# Quick sanity check on the benchmark
benchmark = pd.read_csv(BENCHMARK_PATH)

print(f'Total articles: {len(benchmark)}')
print(f"Should keep:    {benchmark['should_keep'].sum()} ({benchmark['should_keep'].mean():.0%})")
print(f"Should discard: {(~benchmark['should_keep']).sum()}")
print()
print('By source:')
print(benchmark.groupby('source')['should_keep'].agg(['sum', 'count', 'mean']).round(2))
print()
print('By confidence:')
print(benchmark['confidence'].value_counts())
print()
print('Low-confidence entries (manual review recommended):')
pd.set_option('display.max_colwidth', 90)
low = benchmark[benchmark['confidence'] == 'low'][['source', 'title', 'should_keep', 'annotation_reason']]
print(low.to_string(index=False))

Total articles: 113
Should keep:    32 (28%)
Should discard: 81

By source:
              sum  count  mean
source                        
ars-technica   28     45  0.62
reddit          4     68  0.06

By confidence:
confidence
98    38
95    18
99    14
88    11
92     9
96     7
97     6
78     3
84     3
34     1
94     1
93     1
86     1
Name: count, dtype: int64

Low-confidence entries (manual review recommended):
Empty DataFrame
Columns: [source, title, should_keep, annotation_reason]
Index: []
